In [1]:
!git clone https://github.com/andreghl/thesis.git
%cd thesis

Cloning into 'thesis'...
remote: Enumerating objects: 612, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 612 (delta 45), reused 120 (delta 33), pack-reused 472 (from 1)
Receiving objects: 100% (612/612), 53.00 MiB | 16.28 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/home/andre/code/thesis/thesis


In [2]:
!pip install -r requirements.txt -q

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.14/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [3]:
# !python main.py

In [4]:
from bargain import generate_observations, train, pretrain
from bargain.networks import GainNN
from bargain.utils import save_params
import bargain.networks as net
import random

seed: int = 0
a, b = 0, 2026
random.seed(seed)
n_runs: int = 3
load_ext: bool = True
total_timesteps: int = 200000


print("Generating observations...")
generate_observations(filename = "data/instances.h5",
                      obs = 1000000,
                      vehicles = 3,
                      customers = 9,
                      radius = 1.0,
                      seed = random.randint(a, b))

print("Generating observations for hyperparameter tuning...")
generate_observations(filename = "data/tune.h5",
                      obs = 100000,
                      vehicles = 3,
                      customers = 9,
                      radius = 1.0,
                      seed = random.randint(a, b))

print("Parameters to tune for the neural network: ")
parameters = {
    "learning_rate": (1e-5, 1e-1),
    "weight_decay": (0.0, 1e-3),
    "batch_size": (32, 180)}

for key, value in parameters.items():
    print(f"> {key}: {value}")

print("Tuning Gain network...")
score, params = net.tune(model = GainNN(),
                         parameters = parameters,
                         n_models = 15,
                         data_path = "data/tune.h5",
                         features = ["instance", "coalitions"],
                         target = "gain",
                         label = "GainNN",
                         tune_epochs = 10,
                         seed = random.randint(a, b))

print(f"Selected parameters: {params} with score {score}")
save_params(params = params,
            score = score,
            model_name = "GainNN")

print("Training Gain network...")

params = {key: value for key, value in params.items() if key not in ['seed', 'score']}
net.train(**params,
          model = GainNN(),
          n_epochs = 30,
          data_path = "data/instances.h5",
          features = ["instance", "coalitions"],
          target = "gain",
          label = "GainNN",
          verbose = 1,
          seed = random.randint(a, b))

Generating observations...


 20%|██        | 202823/1000000 [29:14<1:54:55, 115.61it/s]


KeyboardInterrupt: 

In [ ]:
from google.colab import userdata
import os
token = userdata.get('GITHUB_TOKEN')
os.environ['GITHUB_TOKEN'] = token

!git add .
!git commit -m "Pretraining on Google Colab"
!git push origin colab